In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv(r"C:\Users\sjs93\Downloads\lfp_kinematics_habit_dishabit (1).csv")

In [3]:
# Keep only cagemate interactions

df = df.loc[
    df["condition"] == "cagemate"
].copy()

print(df.shape)

print(df["condition"].value_counts())

(148050, 23)
condition
cagemate    148050
Name: count, dtype: int64


In [4]:
# --------------------------------------------------
# UM hierarchy ranks used for relative-rank analysis
#
# These UM (urine marking) ranks were taken from:
# Just the results from 8/30 - done during habit dishabit
#
# We specifically used the UM hierarchy data collected
# during the Habituation/Dishabituation phase because:
# - it was temporally aligned with the recordings
# - HCO rankings were incomplete for Cage 4
# - it provided complete hierarchy information across cages
#
# Lower numerical rank = more dominant
# --------------------------------------------------

um_rank_map = {
    "1.1": 2,
    "1.2": 1,
    "1.3": 3,

    "2.1": 1,
    "2.2": 2,
    "2.3": 4,
    "2.4": 2,

    "3.1": 2,
    "3.2": 3,
    "3.3": 1,

    "4.1": 3,
    "4.3": 1,
    "4.4": 1
}

## Habituation/Dishabituation Partner Schedule

The `schedule` dictionary maps each subject mouse to:
- the repeating cagemate (`A`)
- the "novel" cagemate (`B`)

used during the Habituation/Dishabituation paradigm in
Phase 1 of Meghan Cum’s SocialMemoryEphys Pilot 2 experiment.

These mappings were manually generated from the experimental
trial schedule spreadsheet: https://uflorida.sharepoint.com/:x:/r/teams/Padilla-CoreanoLab/Shared%20Documents/General/Data/SocialMemoryEphysPilot2/social_mem_ephys_pilot2_schedule%201.xlsx?d=w67ee7b5b2cdf4f59835b9b96c6b1313b&csf=1&web=1&e=gxtdwF

Experimental structure:
- Exposures 1–4 (`A`) = repeating cagemate
- Exposure 5 (`B`) = "novel" cagemate

This mapping was used to:
1. identify the actual partner mouse for each sniff event
2. append partner-specific hierarchy ranks
3. compute subject-relative hierarchy relationships

Example:
- Subject `1.1`
    - repeating cagemate (`A`) = `1.2`
    - "novel" cagemate (`B`) = `1.3`

Relative rank was then calculated using the UM hierarchy
ranks collected during the Habituation/Dishabituation phase.

In [5]:
schedule = {

    "1.1": {"A": "1.2", "B": "1.3"},
    "1.2": {"A": "1.3", "B": "1.1"},
    "1.3": {"A": "1.1", "B": "1.2"},

    "2.1": {"A": "2.4", "B": "2.3"},
    "2.2": {"A": "2.3", "B": "2.4"},
    "2.3": {"A": "2.2", "B": "2.1"},
    "2.4": {"A": "2.1", "B": "2.2"},

    "3.1": {"A": "3.2", "B": "3.3"},
    "3.2": {"A": "3.3", "B": "3.1"},
    "3.3": {"A": "3.1", "B": "3.2"},

    "4.1": {"A": "4.3", "B": "4.4"},
    "4.4": {"A": "4.1", "B": "4.3"},
}

## Standardize Data Types for Mapping

Before mapping hierarchy ranks and social partner identities,
the `subject` and `partner` columns were converted to strings.

In [6]:
df["subject"] = df["subject"].astype(str)
df["partner"] = df["partner"].astype(str)

In [7]:
#Map actual cagemates to the "partner" column (A or B) based on the schedule
def get_partner_mouse(row):

    subj = row["subject"]
    label = row["partner"]

    if subj in schedule and label in ["A", "B"]:

        return schedule[subj][label]

    return np.nan

In [8]:
# Apply the partner-mapping function row-by-row
#
# For each sniff-event row:
# - read the subject mouse ID
# - read the partner label ("A" or "B")
# - use the Habituation/Dishabituation schedule
#   dictionary to identify the actual partner mouse
#
# Example:
# subject = "1.1"
# partner = "A"
#
# -> repeating partner = "1.2"
#
# The resulting partner mouse ID is stored in:
#     df["partner_mouse"]
#
# axis=1 tells pandas to apply the function
# across rows rather than columns.

df["partner_mouse"] = df.apply(
    get_partner_mouse,
    axis=1
)

In [9]:
# Map UM hierarchy ranks onto the dataframe
#
# The um_rank_map dictionary contains the hierarchy rank
# assigned to each mouse during the Habituation/Dishabituation
# phase using the UM assay.
#
# Lower numerical value = more dominant
#
# Example:
# "1.2" -> rank 1 (more dominant)
# "1.3" -> rank 3 (less dominant)
#
# --------------------------------------------------
# Subject hierarchy rank
#
# Maps each subject mouse ID to its corresponding
# UM hierarchy rank and stores the result in:
#     df["subject_um_rank"]
# --------------------------------------------------

df["subject_um_rank"] = (
    df["subject"]
    .map(um_rank_map)
)

# --------------------------------------------------
# Partner hierarchy rank
#
# Maps each social partner mouse ID to its
# corresponding UM hierarchy rank and stores
# the result in:
#     df["partner_um_rank"]
#
# This allows comparison between:
# - subject hierarchy position
# - partner hierarchy position
#
# which is used for downstream relative-rank analysis.
# --------------------------------------------------

df["partner_um_rank"] = (
    df["partner_mouse"]
    .map(um_rank_map)
)

In [10]:
# --------------------------------------------------
# Compute subject-relative hierarchy relationship
#
# This function compares:
# - the subject mouse's UM hierarchy rank
# - the social partner's UM hierarchy rank
#
# to determine whether the partner is:
# - more dominant than the subject
# - less dominant than the subject
# - or equal in rank
#
# IMPORTANT:
# Lower numerical rank = more dominant
#
# Example:
# subject rank = 2
# partner rank = 1
#
# Since 1 < 2:
# the partner is MORE dominant than the subject
#
# -> returns:
#    "higher_than_subject"
#
# --------------------------------------------------

def relative_rank(row):

    # extract subject hierarchy rank
    subj = row["subject_um_rank"]

    # extract partner hierarchy rank
    partner = row["partner_um_rank"]

    # return NaN if either rank is missing
    if pd.isna(subj) or pd.isna(partner):

        return np.nan

    # lower numerical value = more dominant

    # partner more dominant than subject
    if partner < subj:

        return "higher_than_subject"

    # partner less dominant than subject
    elif partner > subj:

        return "lower_than_subject"

    # partner and subject same hierarchy rank
    else:

        return "equal_rank"

In [11]:
# Apply the relative-rank function row-by-row
#
# For each sniff-event row:
# - compare the subject mouse's hierarchy rank
# - compare the partner mouse's hierarchy rank
#
# The function returns a categorical label describing
# the partner's hierarchy status relative to the subject:
#
# - "higher_than_subject"
# - "lower_than_subject"
# - "equal_rank"
#
# The resulting relative-rank classification is stored in:
#     df["relative_rank"]
#
# axis=1 tells pandas to apply the function across rows
# (each row = one sniff event).

df["relative_rank"] = df.apply(
    relative_rank,
    axis=1
)

In [12]:
print(df.columns.tolist())

['Unnamed: 0', 'subject', 'recording', 'event', 'partner', 'trial', 'condition', 'event_length', 'distance', 'velocity_mouse1', 'velocity_mouse2', 'moving_angle_mouse1', 'moving_angle_mouse2', 'front_orientation_mouse1', 'front_orientation_mouse2', 'rump_orientation_mouse1', 'rump_orientation_mouse2', 'body_angle_mouse1', 'body_angle_mouse2', 'band', 'metric', 'region_or_pair', 'value', 'partner_mouse', 'subject_um_rank', 'partner_um_rank', 'relative_rank']


In [13]:
import os
print(os.getcwd())

c:\Users\sjs93\OneDrive - University of Florida\Documents\GitHub\diff_fam_social_memory_ephys


In [14]:
df.to_csv(
    r"SocialMemoryEphysPilot2/rank_analysis/dataframes/lfp_kinematics_habit_dishabit_cagemate_rank.csv",
    index=False
)